In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

# 由地面观测维持的稀疏导航图：HM3Dv2

旧 `Environment` 的二维更新是按世界高度筛选后投影占用；[04](04_hm3dv2_rgbd_mapping.ipynb) 展示了这一过程。
它没有证明机器人足迹下存在地面，也没有验证跨高度变化的连通性。本实验使用 `GroundMap` 替代这部分判断：

1. 从融合点云的局部法向和粗糙度提取潜在支撑面，保留各处高度与同一 XY 下的不同表面。
2. 默认 0.8×0.8 m 足迹，对应 0.4 m 稀疏导航节点和 0.1 m 足迹内部支撑探针；检查整个足迹及移动途中包围区域的支撑、坡度、台阶和碰撞。
3. 当前高度附近的假设置信度为 0.3，机器人近处最高为 0.6；完整观测且经足迹检查连通到当前机器人位置的地面为 1；碰撞或落差证据为 0。
4. 缺少深度、视野外和被遮挡都不是可行证据。有效深度穿过预期地面，或实际看到较低支撑面，才提供空洞/落差的反证。
5. 输出二维置信度/高度/支撑/状态层，向完整三维点云写入 `ground_confidence`、`ground_state` 和 `ground_candidate`。

这里的“置信度”是可解释的证据分数，不是校准概率。`ground_candidate` 仅指局部支撑形状，可能包括桌面；确认可行地面应看 `ground_confidence == 1`。
算法不使用语义真值判定地面。先验证可控的楼梯、空洞、多级地面，再处理两段真实回放；不能把某个 HM3D 房间的表现当作所有地形的验证。

In [ ]:
from scipykit.task.cv import *

import json
import os
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd

from dataset_wrapper.datasets.hm3d import HM3D
from graphmap import Environment, GroundConfig, GroundMap, GroundState
from graphmap.adapters import rgbd_observation
from graphmap.pose import Pose
from graphmap.sensor import CameraIntrinsic

ROOT = Path(os.environ.get("GRAPHMAP_HM3D_ROOT", "/R/Datasets/3DGS/hm3dv2"))
SCENES = os.environ.get("GRAPHMAP_GROUND_SCENES", "0000,0001").split(",")
N_FRAMES = int(os.environ.get("GRAPHMAP_GROUND_FRAMES", "120"))
STRIDE = int(os.environ.get("GRAPHMAP_HM3D_STRIDE", "8"))
WORKERS = int(os.environ.get("GRAPHMAP_HM3D_WORKERS", "2"))
CHECK_EVERY = 10
config = GroundConfig(footprint=(.8, .8), robot_height=1., max_step=.12, max_slope_degrees=15)
camera = CameraIntrinsic.from_sensor(width=640, height=640, hfov=np.deg2rad(64), vfov=np.deg2rad(64))
if N_FRAMES < 20 or STRIDE < 1 or WORKERS < 0:
    raise ValueError("要求至少 20 帧、正像素步长和非负 worker 数")

workdir = Path.cwd()
examples_dir = workdir / "examples" if (workdir / "examples").is_dir() else workdir
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = examples_dir / "outputs" / "ground_navigation" / run_id
output_dir.mkdir(parents=True, exist_ok=True)
figure_options = dict(assets_root=output_dir / "assets", notebook_name="05_hm3dv2_ground_navigation")
print("Python:", sys.executable)
print("Run:", output_dir.resolve())
print("Ground configuration:", asdict(config))

## 1. 先检验几何机制

实验能力参数是可调整的假设：单级台阶 0.12 m、坡度 15°、碰撞高度 1 m；不表示任何特定机器人的实测能力。
0.5 m 是高度先验范围，**不是全局高度裁剪**。一个总高差 0.4 m、由缓坡连接的平台可以通过；0.18 m 高的连续台阶不能通过。
空洞例子包含低 0.5 m 的可见底部，机器人必须绕行。测试与真实回放使用同一个 `GroundMap`，图中只画稀疏节点和测量点。

In [ ]:
synthetic = {}
x, y = np.meshgrid(np.arange(-2.4, 3.21, .05), np.arange(-1.6, 1.61, .05), indexing="ij")
for name in ["stairs", "hole", "connected_levels"]:
    if name == "stairs":
        z = np.clip(np.floor((x + .2) / .3), 0, 4) * .18
        goal = [2.4, 0, .72]
    elif name == "hole":
        z = np.where((x >= .6) & (x <= 1.4) & (np.abs(y) <= .35), -.5, 0.)
        goal = [2.4, 0, 0]
    else:
        z = np.clip((x + .4) / 2, 0, 1) * .4
        goal = [2.4, 0, .4]
    points = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    nav = GroundMap(config).update(points, robot_pose=Pose([-1.2, 0, 0]), ground_z=0)
    path = nav.plan_path([-1.2, 0, 0], goal)  # 默认只允许 confidence=1 的节点和边
    assert (len(path) == 0) if name == "stairs" else (len(path) > 1)
    if name == "hole":
        assert np.max(np.abs(path[:, 1])) >= .8
    if name == "connected_levels":
        assert np.ptp(path[:, 2]) > .39
    synthetic[name] = (points, nav, path)

fig = mfigure((13, 4.8), dpi=150, alpha=0)
for col, (name, (points, nav, path)) in enumerate(synthetic.items()):
    ax = fig.add_axes([.01 + col / 3, .22, .30, .65], projection="3d")
    ax.scatter(*points[::4].T, c="0.7", s=1.5, alpha=.4, linewidths=0)
    artist = ax.scatter(*nav.xyz.T, c=nav.confidence, vmin=0, vmax=1, cmap="viridis", s=8, depthshade=False)
    if len(path):
        ax.plot(*path.T, c="#d52d87", lw=2)
    ax.set(title=name.replace("_", " "), xlabel="World x (m)", ylabel="World y (m)", zlabel="z (m)")
    ax.set_box_aspect([5.6, 3.2, 1.6])
    ax.locator_params(axis="z", nbins=2)
    ax.view_init(elev=40, azim=-70)
cax = fig.add_axes([.29, .13, .42, .025])
fig.colorbar(artist, cax=cax, orientation="horizontal", label="Ground navigation confidence")
disp(fig, "01_stairs_hole_connected_levels", **figure_options)
plt.close(fig)
print("PASS: stairs rejected; hole bypassed; 0.4 m connected levels traversed")

## 2. 盲区先验、反证与重新观测

相机向下看：没有有效深度时，机器人附近仍是有限先验；当有效射线穿过预期地面、看到更远处时，记录落差证据。
后续空帧不能清除它。实际看到地面后，才可恢复支撑并提升到 1。
这是主动观测的闭环入口：`observation_targets()` 返回需要补看的世界系地面位置；相机朝向或机器人移动由调用方控制，离线回放不会凭空生成新视角。

In [ ]:
down_camera = CameraIntrinsic.from_sensor(width=64, height=64, fx=32, fy=32, ppx=32, ppy=32)
down_pose = Pose([0, 0, 1], rotation=[0, 90, 0])
probe = GroundMap(config)
depth_args = dict(robot_pose=Pose(), ground_z=0, intrinsic=down_camera, pose_camera=down_pose)
transitions = []
for name, depth_value in [("blind / invalid depth", 0.), ("ray passes floor", 2.), ("invalid after hole", 0.)]:
    probe.update(np.empty((0, 3)), depth=np.full((64, 64), depth_value), **depth_args)
    transitions.append(dict(observation=name, confidence=probe.robot_confidence, state=GroundState(probe.robot_state).name))
flat_floor = np.column_stack([x.ravel(), y.ravel(), np.zeros(x.size)])
probe.update(flat_floor, depth=np.ones((64, 64)), **depth_args)
transitions.append(dict(observation="measured floor restored", confidence=probe.robot_confidence, state=GroundState(probe.robot_state).name))
assert [row["confidence"] for row in transitions] == [.6, 0., 0., 1.]
disp(pd.DataFrame(transitions))

## 3. 两段 HM3D RGB-D 回放

`0000`、`0001` 是 dataset_wrapper 的 episode 目录编号，不代表两个独立物理场景。这里检验不同回放观测下的结果。

仍使用已验证的 640×640、64° FOV、毫米 axial 深度与 graphmap 相机坐标。
**地面导航必须使用机器人机体位姿**：由回放相机位姿 `T_world_camera` 和元数据安装外参 `T_body_camera` 恢复
`T_world_body = T_world_camera * ~T_body_camera`。相机高度不能直接作为地面高度。
这里使用已验证导出的机体基准 Z 作为机器人处地面高度；其他机器人应提供轮/足接触面与机体原点之间的偏移。

每段回放沿整个导出序列均匀取 120 帧，逐帧融合 RGB-D/语义，每十帧重新评估导航图，并加入**该检查帧**的深度支撑反证。
这是离线实验的计算折中：中间帧参与三维融合，但其负地面证据未逐帧处理。在线接入时应按所需频率更新。
时间戳仍用序号 × 0.1 秒，只表达先后顺序。

回放中的位姿不保证容纳 0.8 m 机器人，检查失败会原样记录。为展示已确认路线，最终从检查点中选择连通分量最大的参考位姿，
在完整融合云上重新评估；它不是对原始整条回放轨迹的通行保证。

In [ ]:
def sensor_and_body_poses(sample):
    frames = sample["metadata"]["coordinate_frames"]
    assert frames["export_pose"] == "floor_relative_rgb_camera"
    pose = sample["annotations"]["pose"]
    T_world_camera = Pose(pose["position"], pose["quaternion_xyzw"], wrd_frame="world", ego_frame="camera")
    T_body_camera = Pose(frames["camera_local_position"], frames["camera_local_quaternion_xyzw"],
                         wrd_frame="body", ego_frame="camera")
    return T_world_camera, T_world_camera * ~T_body_camera


def assess(nav, env, sample):
    camera_pose, body_pose = sensor_and_body_poses(sample)
    return nav.update(env.infopoints.xyz, robot_pose=body_pose, ground_z=body_pose.position[2],
                      depth=sample["data"]["depth"], depth_scale=.001, depth_range=(.15, 8),
                      intrinsic=camera, pose_camera=camera_pose)


def replay_scene(scene):
    scene_dir = output_dir / scene
    scene_dir.mkdir()
    ds = HM3D(ROOT, episodes=[scene], load_replay=True)
    selected = np.linspace(0, len(ds) - 1, min(N_FRAMES, len(ds)), dtype=int)
    for index in selected:
        assert ds.record(int(index)).metadata["config_sha256"]["simulator_config"] == "7570bfa76cd07c69eee812f2b170ac0ef492a190c617e0eb897d797be7617bec"
    env, nav = Environment(resolution=.05), GroundMap(config)
    rows, body_path, manifest = [], [], []
    started = perf_counter()
    with ds.iloc[selected].loader(num_workers=WORKERS, batch_size=None, prefetch=2) as samples:
        for order, sample in enumerate(samples):
            frame = sample["data"]
            camera_pose, body_pose = sensor_and_body_poses(sample)
            body_path.append(body_pose.position)
            manifest.append(dict(dataset_index=int(selected[order]), frame=sample["key"][1]))
            observation = rgbd_observation(
                frame["depth"], intrinsic=camera, pose_camera=camera_pose, pose_robot=body_pose,
                rgb=frame["rgb"], attributes={} if frame["semantic"] is None else {"semantic_id": frame["semantic"]},
                depth_scale=.001, depth_range=(.15, 8), stride=STRIDE, timestamp=order * .1,
            )
            env.observe(**observation)
            if (order + 1) % CHECK_EVERY == 0 or order + 1 == len(selected):
                tick = perf_counter()
                assess(nav, env, sample)
                rows.append(dict(
                    fused_frames=order + 1, dataset_index=int(selected[order]), frame=sample["key"][1],
                    map_points=env.N, nodes=len(nav.xyz), confirmed_nodes=int(np.count_nonzero(nav.confidence == 1)),
                    blocked_nodes=int(np.count_nonzero(nav.confidence == 0)),
                    robot_confidence=nav.robot_confidence, robot_support=nav.robot_support,
                    robot_state=GroundState(nav.robot_state).name, navigation_seconds=perf_counter() - tick,
                ))
                print(scene, sample["key"][1], "points", env.N, "confirmed", rows[-1]["confirmed_nodes"],
                      "robot", rows[-1]["robot_state"], flush=True)
    elapsed = perf_counter() - started
    stats = pd.DataFrame(rows)
    stats.to_csv(scene_dir / "online_checkpoints.csv", index=False)
    pd.DataFrame(manifest).to_csv(scene_dir / "frame_manifest.csv", index=False)
    nav.save(scene_dir / "online_ground.npz")

    # 比较最有希望的三个已检查位置，保留所有检查结果，避免暗示原轨迹全部可行。
    ranked = stats.sort_values(["confirmed_nodes", "robot_confidence"], ascending=False).head(3)
    reference_rows, best, best_count = [], None, -1
    for row in ranked.itertuples():
        sample = ds.loc[scene, row.frame]
        candidate = GroundMap.load(scene_dir / "online_ground.npz")
        assess(candidate, env, sample)
        count = int(np.count_nonzero(candidate.confidence == 1))
        reference_rows.append(dict(frame=row.frame, confirmed_nodes=count,
                                   robot_state=GroundState(candidate.robot_state).name))
        if count > best_count:
            best, best_count = (candidate, sample), count
    nav, reference = best
    pd.DataFrame(reference_rows).to_csv(scene_dir / "reference_checks.csv", index=False)
    path = np.empty((0, 3))
    if len(nav.roots):
        start = nav.xyz[nav.roots[0]]
        connected = nav.xyz[nav.confidence == 1]
        goal = connected[np.argmax(np.linalg.norm(connected - start, axis=1))]
        path = nav.plan_path(start, goal)
        assert len(path) > 0
    attributes = nav.point_attributes(env.infopoints.xyz)
    env.set_point_attributes(attributes)  # 刷新旧点和新点，语义表保持不变
    env.set_map2d(nav.to_map2d())  # 派生图不再自动接受旧的高度投影更新
    env.save(scene_dir / "environment")
    nav.save(scene_dir / "ground.npz")
    np.save(scene_dir / "confirmed_path.npy", path)
    restored, restored_nav = Environment.load(scene_dir / "environment"), GroundMap.load(scene_dir / "ground.npz")
    np.testing.assert_array_equal(env.infopoints.data.view(np.uint32), restored.infopoints.data.view(np.uint32))
    np.testing.assert_array_equal(env.map2d.grid, restored.map2d.grid)
    for name, values in attributes.items():
        np.testing.assert_array_equal(values, restored.get_index_values(name))
    assert env.get_index_values("semantic_id") == restored.get_index_values("semantic_id")
    if len(path):
        np.testing.assert_array_equal(path, restored_nav.plan_path(path[0], path[-1]))
    summary = dict(
        scene=scene, frames=len(selected), cloud_points=env.N, reference_frame=reference["key"][1],
        sparse_nodes=len(nav.xyz), confirmed_nodes=best_count,
        confirmed_cloud_points=int(np.count_nonzero(attributes["ground_confidence"] == 1)),
        blocked_nodes=int(np.count_nonzero(nav.confidence == 0)),
        uncertain_nodes=int(np.count_nonzero((nav.confidence > 0) & (nav.confidence < 1))),
        path_vertices=len(path), path_length_m=float(np.linalg.norm(np.diff(path, axis=0), axis=1).sum()),
        checkpoints=len(stats), checkpoints_with_confirmed_stance=int(np.count_nonzero(stats.robot_confidence == 1)),
        replay_wall_seconds=elapsed, median_navigation_seconds=float(stats.navigation_seconds.median()),
        config=asdict(config), input=dict(root=str(ROOT), stride=STRIDE, depth_scale=.001, depth_range=[.15, 8]),
        validation=dict(cloud_attributes=True, persistence=True, confirmed_path=bool(len(path))),
    )
    (scene_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n")
    return dict(env=env, nav=nav, path=path, stats=stats, attributes=attributes,
                body_path=np.asarray(body_path), summary=summary)


results = {scene: replay_scene(scene) for scene in SCENES}
disp(pd.DataFrame([result["summary"] for result in results.values()])[
    ["scene", "frames", "cloud_points", "sparse_nodes", "confirmed_nodes", "confirmed_cloud_points",
     "path_length_m", "checkpoints_with_confirmed_stance", "checkpoints"]])

## 4. 二维置信度与三维点云使用同一份结果

二维数组直接显示 `grid[h,w]`，横轴世界 y，纵轴世界 x，不转置。灰色为没有节点的区域；紫色 0 为被拒绝的足迹位置，
中间分数是先验/观测不足/未连通，黄色 1 为确认连通地面。粉色路线只使用分数为 1 的节点和边。
它是稀疏图上的固定朝向平移路线，节点间已经检查整个移动包围区域；转向、动力学和控制器不在此例中。

三维图保留完整观测云作灰色背景，将有地面先验或确认地面的原始点着色（显示阈值 0.3）。
假设节点不写入原始三维点云，所有三维置信度都对应真实观测点。对多高度区域，图中二维层优先选择已确认连通表面，否则选当前地面高度最近的表面；
完整 3D 图仍保存各个高度，路线搜索不在二维投影上跨层跳跃。

In [ ]:
for scene, result in results.items():
    env, nav, path = result["env"], result["nav"], result["path"]
    grid = env.map2d
    h, w = grid.shape[:2]
    x_edges = (np.array([-.5, h-.5]) - grid.origin[0]) * grid.resolution
    y_edges = (np.array([-.5, w-.5]) - grid.origin[1]) * grid.resolution
    fig = mfigure((12, 5.2), dpi=150, alpha=0)
    ax = fig.add_axes([.07, .15, .35, .73])
    ax.set_facecolor("#dddddd")
    confidence_cmap = get_cmap("viridis").copy()
    confidence_cmap.set_bad("#dddddd")
    image = np.ma.masked_less(grid.grid[:, :, 0], 0)
    artist = ax.imshow(image, origin="lower", extent=[*y_edges, *x_edges], vmin=0, vmax=1,
                        interpolation="nearest", cmap=confidence_cmap)
    if len(path):
        ax.plot(path[:, 1], path[:, 0], color="#e52e93", lw=2, label="Confirmed path")
    ax.scatter(nav.robot_position[1], nav.robot_position[0], marker="*", s=90, c="white",
               edgecolors="black", label="Reference robot")
    targets = nav.observation_targets(limit=8)
    if len(targets):
        ax.scatter(targets[:, 1], targets[:, 0], marker="x", s=20, c="#ed7040", label="Look-at targets")
    ax.set_title(f"{scene}: footprint confidence")
    label_axes(ax, "World y (m) / grid w", "World x (m) / grid h")
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.17), ncol=2, fontsize=8)
    cax = fig.add_axes([.44, .25, .012, .49])
    fig.colorbar(artist, cax=cax, label="Confidence")

    ax3d = fig.add_axes([.52, .1, .45, .80], projection="3d")
    points = env.infopoints.xyz
    background = points[np.linspace(0, env.N - 1, min(25000, env.N), dtype=int)]
    ax3d.scatter(*background.T, c="0.55", alpha=.45, s=1.8, linewidths=0)
    confidence = result["attributes"]["ground_confidence"]
    rows = np.flatnonzero(confidence >= .3)
    if len(rows):
        rows = rows[np.linspace(0, len(rows)-1, min(30000, len(rows)), dtype=int)]
        ax3d.scatter(*points[rows].T, c=confidence[rows], vmin=0, vmax=1, cmap="viridis",
                     s=2, linewidths=0, depthshade=False)
    if len(path):
        ax3d.plot(path[:, 0], path[:, 1], path[:, 2] + .05, color="#e52e93", lw=2)
    ax3d.set(title="Full 3D cloud + ground confidence >= 0.3",
             xlabel="World x (m)", ylabel="World y (m)", zlabel="World z (m)")
    ax3d.set_box_aspect(np.maximum(np.ptp(points, axis=0), 1))
    ax3d.locator_params(axis="z", nbins=2)
    ax3d.view_init(elev=56, azim=-60)
    disp(fig, f"02_{scene}_ground_confidence_and_cloud", **figure_options)
    plt.close(fig)

## 5. 查看未通过的位置，决定下一次观测

完整支撑只是在选定空间采样尺度上的证据。空洞小于探针间距/支撑邻域时可能无法分辨；要检测更小危险，必须减小 `support_spacing/support_radius` 并提高输入密度。
`DROP` 表示对当前候选支撑高度的落差/反证，可能是地面洞，也可能是在桌面或天花板候选层上的缺失支撑，不能把计数直接当成场景空洞数量。
同理，几何节点计数不是面积真值。

下面保留回放检查点状态，显示原轨迹是否能容纳当前足迹。空间碰撞失败时应换路线；观测不足时可以使用 `observation_targets()` 的位置生成新的相机观察方向。
盲区假设不会因为重复读取同一空帧而自动提高置信度，也不会覆盖已经看到的障碍物。

In [ ]:
for scene, result in results.items():
    print("Scene", scene, "reference frame", result["summary"]["reference_frame"])
    disp(result["stats"][["frame", "robot_state", "robot_confidence", "robot_support", "confirmed_nodes"]])
    print("Next support observations (world XYZ):")
    print(np.round(result["nav"].observation_targets(limit=6), 3))

fig = mfigure((10, 3.5), dpi=150, alpha=0)
ax1 = fig.add_axes([.08, .20, .37, .68])
ax2 = fig.add_axes([.58, .20, .38, .68])
for scene, result in results.items():
    stats = result["stats"]
    ax1.plot(stats.fused_frames, stats.confirmed_nodes, marker="o", label=scene)
    ax2.plot(stats.fused_frames, stats.navigation_seconds, marker="o", label=scene)
label_axes(ax1, "Fused frames", "Confirmed connected nodes", grid=True, legend=True)
label_axes(ax2, "Fused frames", "Navigation reassessment (s)", grid=True, legend=True)
disp(fig, "03_online_coverage_and_compute", **figure_options)
plt.close(fig)
print("All data, maps, graph snapshots and figures:", output_dir.resolve())

## 接到自己的传感器与导航器

```python
env.observe(**observation)
ground.update(env.infopoints.xyz, robot_pose=T_world_body, ground_z=contact_height,
              depth=depth, intrinsic=camera, pose_camera=T_world_camera,
              depth_scale=0.001, depth_kind="axial")
env.set_point_attributes(ground.point_attributes(env.infopoints.xyz))
env.set_map2d(ground.to_map2d())
```

点云、LiDAR 输入可省略 depth，仍能检查支撑覆盖与可见下层地面；没有地面返回的区域会保持不确定。
已有可靠法向可以通过 `normals=` 提供，并应与输入点逐行对应。语义可以继续用独立 IndexDB 表，不参与这里的几何证明。
三维点属性和二维图是派生结果：融合新观测、换参考位姿或改机器人参数后，应重新计算后再交给规划器。

`env.save` 保存原始点、属性和二维派生图；`ground.save` 另存图结构、机器人配置与负支撑证据，均在本例执行了加载验证。
已有点云融合器暂未删除动态障碍的旧点，所以本版本面向静态场景；动态障碍清理仍需单独实现。
自动法向只给几何候选，未建模材质、承重或语义类别。固定朝向的二维足迹路线不替代 SE(2) 转向或完整机器人运动规划。